# HUDM 5001 — Programming for Data Science
## Session 03 · Python Intro: pandas
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cgpan/humd5001_mini/blob/main/03_Pandas.ipynb)

[**Chenguang Pan**](https://cpan.ai/), [**Youmi Lab**](https://youmilab.ai/)
**Teachers College, Columbia University**

This notebook is used in a mini-session designed to help interested learners quickly grasp fundamental Python skills. For students of HUDM5001 in Fall 2026 semester, please look for [this repo](https://github.com/cgpan/hudm5001) for class materials.

This notebook is based on Prof. Youmi Suk's original design for Programming for Data Science. Original materials can be found [here](https://github.com/youmilab/hudm5001).

> ▶️ **How to open this notebook in Google Colab** — pick whichever is easiest:
> 1. **From Google Drive:** put this `.ipynb` in your Drive, right-click it → *Open with* → *Google Colaboratory*.
> 2. **Upload:** go to [colab.research.google.com](https://colab.research.google.com) → *File ▸ Upload notebook* → choose this file.
> 3. **From GitHub:** click the **Open in Colab** badge above (works once the notebook is pushed to the course repo).
>
> Then **run the Setup cell first** (▶ or `Shift+Enter`). Everything else runs top-to-bottom.

One library today, and it is the one you will reach for most. **pandas** gives you the **DataFrame**: a labeled table, a bit like a spreadsheet you can drive with code. It sits right on top of the NumPy arrays from last week and adds *names* to the rows and columns — so your data stops being anonymous numbers and starts being, say, the petal length of flower number 42.

---
## 1. Big Picture

### 🎯 Learning Objectives
By the end of this session you will be able to:
- Explain what a **Series** and a **DataFrame** are, and how they relate to NumPy arrays.
- Build a DataFrame from a dictionary and **read one from a CSV** (local or a URL).
- Inspect a table with `.head()`, `.shape`, `.columns`, `.dtypes`, and `.describe()`.
- Select columns and rows with `[]`, **`.loc`** (labels) and **`.iloc`** (positions).
- **Filter** rows with boolean masks, combining conditions with `&` and `|`.
- Add new columns, and summarize with `value_counts()` and **`groupby()`**.
- Spot and handle **missing values**, and save your work back to a CSV.

### ✅ Prerequisites
- Sessions 01–02 (data types, operators, and especially NumPy boolean indexing).
- Run the Setup cell to import pandas and set the data URL.

### 🧩 Key Concepts
`Series`  `DataFrame`  `index`  `read_csv`  `.loc`  `.iloc`  `boolean filtering`  `new columns`  `value_counts`  `groupby`  `missing data`

### 📖 Reading
- McKinney, *Python for Data Analysis*, Chapter 5 (Getting Started with pandas)

### 🗺️ Today's Agenda
1. From NumPy to pandas: why labels?
2. The Series — a labeled 1-D column
3. The DataFrame — build and inspect a table
4. Reading real data: the iris dataset
5. Selecting columns and rows (`[]`, `.loc`, `.iloc`)
6. Filtering rows with boolean masks
7. Creating new columns
8. Summaries: `value_counts` and `groupby`
9. Missing data, and saving to CSV

---
## ▶️ Setup — run this cell first
This imports the libraries and sets `IRIS_URL`, the web address of the iris data in the course repo.

In [ ]:
# === SETUP — run me first! ===
import numpy as np
import pandas as pd

# The iris dataset lives in the public course repo, so we can read it straight from the web
# (no download needed) — the same URL trick we used for the wine data in Session 02.
IRIS_URL = "https://raw.githubusercontent.com/cgpan/humd5001_mini/main/assets/iris.csv"
print("pandas", pd.__version__, "| NumPy", np.__version__)
print("Setup complete ✅")

---
## 2. Content

### 2.1 From NumPy to pandas — why labels?
A NumPy array is fast, but it is **positional**: the numbers have no names, so *you* have to remember that position 0 is Ann and position 1 is Bob. pandas keeps NumPy's speed and adds a **label** to every value. That small change is what turns a pile of numbers into a readable table.

In [ ]:
scores = np.array([88, 92, 79])
print(scores)                 # fast, but which number belongs to whom?

labeled = pd.Series([88, 92, 79], index=["Ann", "Bob", "Cara"])
print(labeled)                # same three numbers, now with names attached

### 2.2 The Series — a labeled 1-D column
A **Series** is a one-dimensional sequence of values with an **index** (the labels) running alongside it. Think of it as a single column of a spreadsheet. Underneath, the values are still a NumPy array, so everything you learned about vectorized math and boolean indexing carries straight over.

In [ ]:
temps = pd.Series([68, 71, 65, 74], index=["Mon", "Tue", "Wed", "Thu"])
print(temps)
print("values:", temps.values)      # a plain NumPy array underneath
print("index :", list(temps.index))
print("dtype :", temps.dtype)

You can reach a value by its **label** or by its **position**, and — just like NumPy — filter with a boolean mask.

In [ ]:
print("by label   :", temps["Wed"])     # 65
print("by position:", temps.iloc[0])     # 68  (the first value)
print("mean:", temps.mean(), "| max:", temps.max())
print("warmer than 70:")
print(temps[temps > 70])                 # boolean filtering, exactly like NumPy

Hold onto this idea: **a column of a DataFrame is just a Series.** Once you are comfortable here, a whole table is only a handful of Series lined up side by side.

#### ✏️ Try It Yourself — Exercise 1
**Difficulty:** ★★☆☆☆  **Skills:** `pd.Series`, `.mean()`, boolean filtering

Make a Series of five daily step counts with the weekday names as its index. Print the **average**, then use a boolean mask to show only the days you walked **more than the average**.

In [ ]:
# Your code here


<details>
<summary>▶ Show solution</summary>

```python
steps = pd.Series([8200, 10500, 6400, 12000, 9300],
                  index=["Mon", "Tue", "Wed", "Thu", "Fri"])
print("Average:", steps.mean())
print(steps[steps > steps.mean()])   # only the above-average days
```

</details>

### 2.3 The DataFrame — build and inspect a table
A **DataFrame** is a 2-D table: named columns, and an index down the side. The most common way to build one by hand is from a **dictionary**, where each key is a column name and each value is the list of entries in that column. (Yes — the very same dictionaries from Session 01, now doing heavier lifting.)

In [ ]:
students = pd.DataFrame({
    "name":  ["Maya", "Liam", "Noah"],
    "major": ["Applied Stats", "Learning Analytics", "Measurement"],
    "gpa":   [3.8, 3.5, 3.9],
})
print(students)
print("shape  :", students.shape)          # (rows, columns)
print("columns:", list(students.columns))
print("one column is a", type(students["gpa"]))   # a Series!

#### ✏️ Try It Yourself — Exercise 2
**Difficulty:** ★★☆☆☆  **Skills:** `pd.DataFrame`, `.shape`, `.columns`, `.head()`

Build a DataFrame of **four** movies with three columns: `title`, `year`, and `rating`. Print its **shape** and its **column names** as a list, then show just the **first two rows** with `.head(2)`.

In [ ]:
# Your code here


<details>
<summary>▶ Show solution</summary>

```python
movies = pd.DataFrame({
    "title":  ["Inception", "Matrix", "Titanic", "Up"],
    "year":   [2010, 1999, 1997, 2009],
    "rating": [8.8, 8.7, 7.9, 8.3],
})
print("shape:", movies.shape)
print("columns:", list(movies.columns))
print(movies.head(2))
```

</details>

### 2.4 Reading real data — the iris dataset
Typing tables by hand gets old fast. In practice you **read** data from a file. Here is the famous **iris** dataset: 150 flowers from three species, with four measurements each. You met it in the Session 02 assignment through NumPy — watch how much friendlier it feels with pandas.

`pd.read_csv` happily takes a web address, so there is nothing to download.

In [ ]:
iris = pd.read_csv(IRIS_URL)
print("shape:", iris.shape)
print("columns:", list(iris.columns))
iris.head()

Two habits worth forming: check the **column types** with `.dtypes`, and get a fast statistical snapshot with `.describe()`. Notice `.describe()` quietly skips the text column and summarizes only the numbers.

In [ ]:
print(iris.dtypes)
iris.describe()

### 2.5 Selecting columns and rows
Three tools do almost all of the work:
- **`df["col"]`** — one column, returned as a **Series**. A list of names, `df[["a", "b"]]`, gives a smaller **DataFrame**.
- **`.iloc[...]`** — rows (and columns) by **integer position**, like NumPy. The right end is *excluded*.
- **`.loc[...]`** — rows (and columns) by **label**. Here the right end is *included* — the one place beginners get tripped up, so watch the next cell closely.

In [ ]:
# one column -> a Series ; two-plus columns -> a DataFrame
print(iris["species"].head(3))
print("---")
print(iris[["petal_length", "species"]].head(3))

In [ ]:
# rows by POSITION with .iloc  ->  positions 0,1,2  (the 3 is left out)
print(iris.iloc[0:3])
print("iloc[0:3] returned", iris.iloc[0:3].shape[0], "rows")

# rows by LABEL with .loc  ->  labels 0,1,2,3  (the 3 IS included!)
print(iris.loc[0:3])
print("loc[0:3] returned", iris.loc[0:3].shape[0], "rows")

That off-by-one difference is on purpose: positions are like a ruler (stop *before* the mark), while labels are names (include the one you asked for). You can also grab specific rows *and* columns at once:

In [ ]:
# rows 0-2, only these two columns
print(iris.loc[0:2, ["sepal_length", "species"]])

#### ✏️ Try It Yourself — Exercise 3
**Difficulty:** ★★★☆☆  **Skills:** `.iloc`, `.loc`, column selection

From `iris`: (a) print the **first 3 rows** with `.iloc`; (b) print the `species` of the flower at **label 100** with `.loc`; (c) print the `petal_length` and `petal_width` columns for **rows 0 through 2** with `.loc`.

In [ ]:
# Your code here


<details>
<summary>▶ Show solution</summary>

```python
print(iris.iloc[0:3])
print(iris.loc[100, "species"])
print(iris.loc[0:2, ["petal_length", "petal_width"]])
```

</details>

### 2.6 Filtering rows with boolean masks
This is the move you will use every single day. A comparison on a column produces a **boolean mask** (a Series of `True`/`False`), and putting that mask inside `df[...]` keeps only the `True` rows. It is the exact same idea as NumPy boolean indexing from Session 02, just on a table.

In [ ]:
# a mask: which flowers have petals longer than 5 cm?
mask = iris["petal_length"] > 5
print(mask.head())

# use the mask to keep only those rows
big = iris[iris["petal_length"] > 5]
print("flowers with long petals:", big.shape[0])

To combine conditions, pandas uses **`&`** for *and* and **`|`** for *or* — not the words `and`/`or`, which only work on single `True`/`False` values. Wrap **each condition in parentheses**, because `&` would otherwise grab the numbers before the comparison does.

In [ ]:
# setosa flowers whose petals are on the longer side for that species
setosa_big = iris[(iris["species"] == "setosa") & (iris["petal_length"] > 1.5)]
print("large-petal setosa:", setosa_big.shape[0])
print(setosa_big.head())

#### ✏️ Try It Yourself — Exercise 4
**Difficulty:** ★★★★☆  **Skills:** boolean masks, `&`, `.shape`

How many **virginica** flowers have a `sepal_length` of **at least 7.0**? Build a mask with two conditions joined by `&`, then count the matching rows with `.shape[0]`.

In [ ]:
# Your code here


<details>
<summary>▶ Show solution</summary>

```python
big_virginica = iris[(iris["species"] == "virginica") & (iris["sepal_length"] >= 7.0)]
print("count:", big_virginica.shape[0])
```

</details>

### 2.7 Creating new columns
You add a column by assigning to a new name. The arithmetic runs on the **whole column at once** (vectorized — no loop in sight). And a comparison gives you a `True`/`False` column, which is how we *label* rows without writing an `if`.

In [ ]:
# a new column from arithmetic on existing ones
iris["petal_area"] = iris["petal_length"] * iris["petal_width"]
print(iris[["petal_length", "petal_width", "petal_area"]].head(3))

# a True/False column from a comparison
iris["big_petal"] = iris["petal_length"] > iris["petal_length"].mean()
print(iris[["petal_length", "big_petal"]].head(3))

### 2.8 Summaries: `value_counts` and `groupby`
Two workhorses. **`value_counts()`** tallies how often each value shows up in a column — perfect for categories. **`groupby()`** is the famous *split–apply–combine*: split the rows into groups, compute something on each group, and stack the answers back up. "Average petal length **per species**" is one short line.

In [ ]:
# how many flowers of each species? (counts, then proportions)
print(iris["species"].value_counts())
print(iris["species"].value_counts(normalize=True).round(2))

In [ ]:
# split-apply-combine: mean petal_length within each species
print(iris.groupby("species")["petal_length"].mean())

# group means for a couple of columns at once
print(iris.groupby("species")[["sepal_length", "petal_length"]].mean().round(2))

Sorting rounds out the toolkit — `sort_values` orders the table by a column.

In [ ]:
# the five flowers with the longest petals
print(iris.sort_values("petal_length", ascending=False).head(5))

#### ✏️ Try It Yourself — Exercise 5
**Difficulty:** ★★★★★  **Skills:** `groupby`, `value_counts`, `sort_values`

Answer three questions about `iris`: (a) what is the **average `sepal_width` for each species** (`groupby`)? (b) how **many flowers are in each species** (`value_counts`)? (c) which **three flowers have the largest `sepal_length`** overall (`sort_values` + `.head(3)`)?

In [ ]:
# Your code here


<details>
<summary>▶ Show solution</summary>

```python
print(iris.groupby("species")["sepal_width"].mean())
print(iris["species"].value_counts())
print(iris.sort_values("sepal_length", ascending=False).head(3))
```

</details>

### 2.9 Missing data, and saving your work
Real datasets have holes. pandas marks a missing value as **`NaN`** ("not a number"). Three moves cover most of the early going: **spot** them with `.isna().sum()`, **drop** the affected rows with `.dropna()`, or **fill** them with `.fillna(value)`.

In [ ]:
demo = pd.DataFrame({
    "name":  ["Ann", "Bob", "Cara"],
    "score": [90, np.nan, 88],
})
print(demo)
print("missing per column:")
print(demo.isna().sum())
print("drop the row with a hole:")
print(demo.dropna())
print("or fill it with the column mean:")
print(demo.fillna(demo["score"].mean()))

When you are done, write the table back out with **`to_csv`**. Passing `index=False` keeps pandas from adding a column of row numbers.

In [ ]:
iris.to_csv("iris_out.csv", index=False)
print("re-read shape:", pd.read_csv("iris_out.csv").shape)   # confirm it saved

---
## 3. Conclusions & Key Takeaways
- A **Series** is one labeled column; a **DataFrame** is a table of Series sharing an index.
- Read data with **`pd.read_csv`** (a file path *or* a URL) and inspect it with `.head()`, `.shape`, `.dtypes`, `.describe()`.
- **`.iloc`** selects by position (right end *excluded*); **`.loc`** selects by label (right end *included*).
- **Filter** with boolean masks; combine conditions with **`&`** / **`|`** and parenthesize each one.
- Add columns by assignment; a comparison makes a handy `True`/`False` column — no `if` required.
- **`value_counts()`** tallies categories; **`groupby()`** is split–apply–combine in one line.
- Handle gaps with `.isna()`, `.dropna()`, `.fillna()`, and save with **`to_csv(..., index=False)`**.

### 🧾 Quick Reference
| Task | Code |
|---|---|
| read a CSV / URL | `pd.read_csv(path)` |
| first rows / shape | `df.head()`, `df.shape` |
| one column (Series) | `df["col"]` |
| several columns | `df[["a", "b"]]` |
| rows by position | `df.iloc[0:3]` |
| rows by label | `df.loc[0:3]` |
| filter rows | `df[df["x"] > 5]` |
| two conditions | `df[(df.a > 0) & (df.b < 5)]` |
| new column | `df["c"] = df["a"] + df["b"]` |
| count categories | `df["g"].value_counts()` |
| group means | `df.groupby("g")["x"].mean()` |
| missing values | `df.isna().sum()`, `df.dropna()`, `df.fillna(0)` |
| save a CSV | `df.to_csv("out.csv", index=False)` |

### ⏭️ Coming Up Next
**Week 6 — more pandas, and SQLite databases.** We go deeper into reshaping and joining tables, and start pulling data from a real database. Read McKinney Ch 6–8.

### 📌 This Week's Assignment
A pandas assignment on the **wine** dataset — see the `assignment/` folder / GitHub.

> 💡 **Tip:** make a copy in Colab (*File ▸ Save a copy in Drive*) and scribble in the code cells. Poking at a real DataFrame — changing a filter, regrouping — is by far the fastest way to make pandas stick.